In [1]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Embedding

from google.colab import drive


In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
file_path = "/content/drive/MyDrive/text_generation_dataset.txt"

In [4]:
text = open(file_path, 'r', encoding='utf-8').read()

print("===================================")
print("DATASET LOADED SUCCESSFULLY")
print("===================================")

print("\nFirst 500 Characters:\n")

print(text[:500])

print("\n===================================")
print("TOTAL CHARACTERS")
print("===================================")

print(len(text))

DATASET LOADED SUCCESSFULLY

First 500 Characters:

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor

TOTAL CHARACTERS
1115394


In [5]:
characters = sorted(list(set(text)))

# Character to Integer Mapping
char_to_index = {
    char:index
    for index, char in enumerate(characters)
}

# Integer to Character Mapping
index_to_char = {
    index:char
    for index, char in enumerate(characters)
}

print("\n===================================")
print("TOTAL UNIQUE CHARACTERS")
print("===================================")

print(len(characters))


TOTAL UNIQUE CHARACTERS
65


In [7]:
sequence_length = 100

X = []
y = []

for i in range(0, len(text) - sequence_length):

    input_sequence = text[i:i + sequence_length]

    output_character = text[i + sequence_length]

    X.append(
        [char_to_index[char] for char in input_sequence]
    )

    y.append(
        char_to_index[output_character]
    )

# Convert into numpy arrays
X = np.array(X)
y = np.array(y)

print("\n===================================")
print("TRAINING DATA CREATED")
print("===================================")

print("Total Samples:", len(X))


TRAINING DATA CREATED
Total Samples: 1115294


In [8]:
model = Sequential()

# Embedding Layer
model.add(
    Embedding(
        input_dim=len(characters),
        output_dim=128,
        input_length=sequence_length
    )
)

# LSTM Layer
model.add(
    LSTM(256)
)

# Output Layer
model.add(
    Dense(
        len(characters),
        activation='softmax'
    )
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [13]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [20]:
print("\n===================================")
print("MODEL TRAINING STARTED")
print("===================================")

model.fit(
    X,
    y,
    batch_size=128,
    epochs=20
)


MODEL TRAINING STARTED
Epoch 1/20
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 150s 17ms/step - accuracy: 0.5820 - loss: 1.3591
Epoch 2/20
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 202s 17ms/step - accuracy: 0.5893 - loss: 1.3300
Epoch 3/20
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 148s 17ms/step - accuracy: 0.5944 - loss: 1.3093
Epoch 4/20
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 203s 17ms/step - accuracy: 0.5986 - loss: 1.2930
Epoch 5/20
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 149s 17ms/step - accuracy: 0.6018 - loss: 1.2796
Epoch 6/20
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 149s 17ms/step - accuracy: 0.6043 - loss: 1.2686
Epoch 7/20
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 149s 17ms/step - accuracy: 0.6066 - loss: 1.2589
Epoch 8/20
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 149s 17ms/step - accuracy: 0.6093 - loss: 1.2504
Epoch 9/20
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 149s 17ms/step - accuracy: 0.6108 - loss: 1.2437
Epoch 10/20
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 148s 17ms/step - accuracy: 0.6124 - loss: 1.2373
Epoch 11/20
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 149s 17ms/step - accur

In [21]:
def generate_text(seed_text, next_characters=300):

    generated_text = seed_text

    for i in range(next_characters):

        # Take last 100 characters
        input_sequence = generated_text[-sequence_length:]

        # Pad if input is short
        if len(input_sequence) < sequence_length:

            input_sequence = (
                " " * (sequence_length - len(input_sequence))
                + input_sequence
            )

        # Convert characters to integers
        input_sequence = [
            char_to_index.get(char, 0)
            for char in input_sequence
        ]

        input_sequence = np.array(input_sequence)

        input_sequence = np.expand_dims(input_sequence, axis=0)

        # Predict next character
        prediction = model.predict(
            input_sequence,
            verbose=0
        )

        predicted_index = np.argmax(prediction)

        predicted_character = index_to_char[predicted_index]

        generated_text += predicted_character

    return generated_text


In [22]:
print("\n===================================")
print("MODEL SUMMARY")
print("===================================")

model.summary()


MODEL SUMMARY


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 128)       │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 65)             │        16,705 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,257,797 (4.80 MB)

 Trainable params: 419,265 (1.60 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 838,532 (3.20 MB)

In [26]:
print("\n===================================")
print("TEXT GENERATION SYSTEM")
print("===================================")

seed_text = input("\nEnter starting text:\n\n")



TEXT GENERATION SYSTEM

Enter starting text:

once upon a time there was a king 


In [27]:
generated_output = generate_text(seed_text)

In [28]:
print("\n===================================")
print("GENERATED TEXT")
print("===================================\n")

print(generated_output)


GENERATED TEXT

once upon a time there was a king offence
To bear the body that he hath no more than he
That shall be so stir a banishment of the seas.

KING RICHARD III:
What says he did be the head of this the field?

KING RICHARD II:
What says he did be the head of this the field?

KING RICHARD II:
What says he did be the head of this the field?
